# Phase 3: Fine-Tuning with LoRA and Prompt Tuning (Using S3)
In this notebook, we will experiment with **LoRA (testing different target layers)** and **Prompt Tuning**. Data is loaded directly from S3.

In [ ]:
!pip install -q mlflow boto3 transformers torch pandas scikit-learn datasets peft tqdm accelerate python-dotenv s3fs

### 1. Load Credentials & Set Up MLflow

In [ ]:
import os
import mlflow
from dotenv import load_dotenv

# Load environment variables from .env if present
load_dotenv()

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME") or "finance-sentiment-mlflow-artifacts-kavishka"
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI") or "http://13.235.68.121:5000/"

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_DEFAULT_REGION"] = AWS_DEFAULT_REGION

# Connect to your remote MLflow server
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print("Connected to MLflow at:", mlflow.get_tracking_uri())
print("Using S3 bucket:", S3_BUCKET_NAME)
mlflow.set_experiment("Sentiment_FineTuning_Benchmark")

### 2. Read Datasets Directly From S3 & Prepare

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

print(f"Fetching datasets from S3 Bucket: {S3_BUCKET_NAME} ...")
train_df = pd.read_csv(f"s3://{S3_BUCKET_NAME}/train.csv")
val_df = pd.read_csv(f"s3://{S3_BUCKET_NAME}/val.csv")
test_df = pd.read_csv(f"s3://{S3_BUCKET_NAME}/test.csv")
print("✅ Datasets loaded successfully!")

label2id = {'positive': 0, 'negative': 1, 'neutral': 2}
id2label = {0: 'positive', 1: 'negative', 2: 'neutral'}

for df in [train_df, val_df, test_df]:
    df['label'] = df['Sentiment'].map(label2id)

dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df[['Headline', 'label']]),
    'val': Dataset.from_pandas(val_df[['Headline', 'label']]),
    'test': Dataset.from_pandas(test_df[['Headline', 'label']])
})

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['Headline'], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': acc, 'f1_macro': f1}

print("Data tokenization complete!")

### Experiment 1: LoRA (Targeting Attention Layers Only)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import time

model_lora_attn = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)

lora_config_attn = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]
)
model_lora_attn = get_peft_model(model_lora_attn, lora_config_attn)
model_lora_attn.print_trainable_parameters()

training_args_attn = TrainingArguments(
    output_dir="./results_lora_attn",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer_lora_attn = Trainer(
    model=model_lora_attn,
    args=training_args_attn,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

with mlflow.start_run(run_name="LoRA_Attention_Only"):
    start_time = time.time()
    trainer_lora_attn.train()
    train_time = time.time() - start_time
    mlflow.log_metric("train_time_seconds", train_time)
    
    test_results = trainer_lora_attn.evaluate(tokenized_datasets['test'])
    mlflow.log_metric("test_accuracy", test_results['eval_accuracy'])
    mlflow.log_metric("test_f1_macro", test_results['eval_f1_macro'])
    print("Test Results:", test_results)

### Experiment 2: LoRA (Targeting All Linear Layers)

In [ ]:
model_lora_all = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)

lora_config_all = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "key", "value", "dense"]
)
model_lora_all = get_peft_model(model_lora_all, lora_config_all)
model_lora_all.print_trainable_parameters()

training_args_all = TrainingArguments(
    output_dir="./results_lora_all_linear",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer_lora_all = Trainer(
    model=model_lora_all,
    args=training_args_all,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

with mlflow.start_run(run_name="LoRA_All_Linear"):
    start_time = time.time()
    trainer_lora_all.train()
    train_time = time.time() - start_time
    mlflow.log_metric("train_time_seconds", train_time)
    
    test_results = trainer_lora_all.evaluate(tokenized_datasets['test'])
    mlflow.log_metric("test_accuracy", test_results['eval_accuracy'])
    mlflow.log_metric("test_f1_macro", test_results['eval_f1_macro'])
    print("Test Results:", test_results)

### Experiment 3: Prompt Tuning

In [ ]:
from peft import PromptTuningConfig, PromptTuningInit

model_prompt = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)

prompt_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8,
    prompt_tuning_init_text="Classify the financial sentiment of this headline: ",
    tokenizer_name_or_path=model_name,
)

model_prompt = get_peft_model(model_prompt, prompt_config)
model_prompt.print_trainable_parameters()

training_args_prompt = TrainingArguments(
    output_dir="./results_prompt_tuning",
    eval_strategy="epoch",
    learning_rate=3e-2,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    report_to="none"
)

trainer_prompt = Trainer(
    model=model_prompt,
    args=training_args_prompt,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    compute_metrics=compute_metrics,
)

with mlflow.start_run(run_name="Prompt_Tuning"):
    start_time = time.time()
    trainer_prompt.train()
    train_time = time.time() - start_time
    mlflow.log_metric("train_time_seconds", train_time)
    
    test_results = trainer_prompt.evaluate(tokenized_datasets['test'])
    mlflow.log_metric("test_accuracy", test_results['eval_accuracy'])
    mlflow.log_metric("test_f1_macro", test_results['eval_f1_macro'])
    print("Test Results:", test_results)